# Controlled Start(w) Probabilities

Clean execution notebook for the controlled Start(w) paper figures using only the `start_tokens_only` target-string setting. Plot construction lives in `vis.py`; synthetic target-string run loading and table preparation live in `analysis.py` plus the small manifest filter below.


In [ ]:
def _config_list_key(value):
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    try:
        return ",".join(sorted(str(item) for item in value))
    except TypeError:
        return str(value)


def _is_uncapped(value):
    return value in (None, 0)


def _matches_controlled_startw_config(config):
    if not CONTROLLED_STARTW_REQUIRE_FULL_SYNTHETIC_RUN:
        return isinstance(config, dict)
    if not isinstance(config, dict):
        return False
    if config.get("synthetic_root") != CONTROLLED_STARTW_SYNTHETIC_ROOT:
        return False
    if _config_list_key(config.get("synthetic_langs")) != _config_list_key(CONTROLLED_STARTW_EXPECTED_SYNTHETIC_LANGS):
        return False
    if not _is_uncapped(config.get("max_prompts")):
        return False
    if not _is_uncapped(config.get("max_prompts_per_lang")):
        return False
    return True


def _add_target_string_config_columns(manifest):
    out = manifest.copy()
    out["data_source"] = out["config"].map(
        lambda c: c.get("data_source") if isinstance(c, dict) else None
    )
    out["target_string_scoring_mode"] = out["config"].map(
        lambda c: c.get("target_string_scoring_mode", "multi_token_teacher_forced")
        if isinstance(c, dict)
        else None
    )
    out["decoding_lens"] = out["config"].map(
        lambda c: c.get("decoding_lens") if isinstance(c, dict) else None
    )
    out["synthetic_root"] = out["config"].map(
        lambda c: c.get("synthetic_root") if isinstance(c, dict) else None
    )
    out["synthetic_langs_key"] = out["config"].map(
        lambda c: _config_list_key(c.get("synthetic_langs")) if isinstance(c, dict) else ""
    )
    out["max_prompts"] = out["config"].map(
        lambda c: c.get("max_prompts") if isinstance(c, dict) else None
    )
    out["max_prompts_per_lang"] = out["config"].map(
        lambda c: c.get("max_prompts_per_lang") if isinstance(c, dict) else None
    )
    return out

%load_ext autoreload
%autoreload 2

from pathlib import Path
import hashlib
import json
import sys

import matplotlib.pyplot as plt
from matplotlib import font_manager
import pandas as pd
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vis.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis import (
    MAIN_PAPER_SYNTHETIC_MODELS,
    add_normalized_layers,
    discover_eval_runs,
    load_synthetic_target_string_prompt_metrics,
    load_target_string_menu_dataframe,
    select_latest_unique_runs,
    summarize_target_string_menu_entries,
    synthetic_model_display_name,
    translation_target_from_data_source,
)
from vis import (
    plot_startw_copy_cloze_grid,
    plot_startw_translation_target_grid,
    plot_startw_translation_source_grid,
    save_matplotlib_figure_bundle,
    set_matplotlib_paper_font,
)

for font_path in font_manager.findSystemFonts():
    if "avenir" in font_path.lower():
        font_manager.fontManager.addfont(font_path)

AVENIR_CACHED_FACES = {
    REPO_ROOT / ".cache" / "fonts" / "AvenirNext-Medium.ttf": 5,
    REPO_ROOT / ".cache" / "fonts" / "AvenirNext-DemiBold.ttf": 2,
}
if any(not path.exists() for path in AVENIR_CACHED_FACES):
    from fontTools.ttLib import TTCollection

    avenir_next_ttc = next(
        (Path(path) for path in font_manager.findSystemFonts() if Path(path).name == "Avenir Next.ttc"),
        None,
    )
    if avenir_next_ttc is not None:
        collection = TTCollection(avenir_next_ttc)
        for cache_path, face_index in AVENIR_CACHED_FACES.items():
            if not cache_path.exists():
                cache_path.parent.mkdir(parents=True, exist_ok=True)
                collection.fonts[face_index].save(cache_path)
for cache_path in AVENIR_CACHED_FACES:
    if cache_path.exists():
        font_manager.fontManager.addfont(cache_path)

set_matplotlib_paper_font(families=("Avenir", "Avenir Next", "DejaVu Sans"))

LOG_ROOT = REPO_ROOT / "logs" / "evals"
FIG_DIR = REPO_ROOT / "figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)
PAPER_MAIN_FIG_DIR = FIG_DIR / "paper" / "main"
PAPER_APPENDIX_FIG_DIR = FIG_DIR / "paper" / "appendix"
CONTROLLED_STARTW_CACHE_DIR = REPO_ROOT / ".analysis_cache" / "controlled_startw_probs"
CONTROLLED_STARTW_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Top-level knobs for keeping this notebook focused and fast.
CONTROLLED_STARTW_SCORING_MODE = "start_tokens_only"
CONTROLLED_STARTW_MODELS = MAIN_PAPER_SYNTHETIC_MODELS
CONTROLLED_STARTW_LANGUAGE_ORDER = ["en", "fr", "tr", "ru", "ar", "hi", "zh"]
CONTROLLED_STARTW_COPY_CLOZE_DATA_SOURCES = ["copy", "cloze"]
CONTROLLED_STARTW_COPY_CLOZE_LANGS = list(CONTROLLED_STARTW_LANGUAGE_ORDER)
CONTROLLED_STARTW_COPY_CLOZE_MENU_LANGS = None  # None means plot every menu language as a curve.
CONTROLLED_STARTW_TRANSLATION_TARGET_LANGS = list(CONTROLLED_STARTW_LANGUAGE_ORDER)
CONTROLLED_STARTW_TRANSLATION_MENU_LANGS = None  # None means plot every menu language as a curve.
CONTROLLED_STARTW_TRANSLATION_PROMPT_LANGS = None  # Set to e.g. ["fr", "de", "ru"] to filter loaded source prompts.
CONTROLLED_STARTW_TRANSLATION_PROMPT_COLUMN_LANGS = list(CONTROLLED_STARTW_LANGUAGE_ORDER)
CONTROLLED_STARTW_START_TOKEN_VALUE_COL = "menu_start_token_prob"
CONTROLLED_STARTW_START_TOKEN_INCLUDE_VALUES = (
    "menu_start_token_prob",
    "menu_start_token_prob_share",
    "menu_start_token_logprob",
)

CONTROLLED_STARTW_SYNTHETIC_ROOT = "data/wendler_2024_data/common69"
CONTROLLED_STARTW_EXPECTED_SYNTHETIC_LANGS = [
    "ar", "bg", "cs", "de", "en", "es", "fa", "fi", "fr", "gl", "hi", "id", "is",
    "it", "ja", "ko", "mr", "pl", "pt_br", "ru", "sr", "sv", "th", "tr", "uk", "ur", "zh",
]
CONTROLLED_STARTW_REQUIRE_FULL_SYNTHETIC_RUN = True
CONTROLLED_STARTW_USE_CACHE = True
CONTROLLED_STARTW_FORCE_REBUILD_CACHE = False


def _cache_jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (list, tuple)):
        return [_cache_jsonable(item) for item in value]
    if isinstance(value, set):
        return sorted(_cache_jsonable(item) for item in value)
    if isinstance(value, dict):
        return {str(key): _cache_jsonable(val) for key, val in sorted(value.items())}
    return value


def startw_cache_key(manifest, *, label, extra=None):
    manifest_cols = [
        col
        for col in ["data_source", "display_model_name", "model_name", "revision", "exp_id", "run_mtime_ns"]
        if col in manifest.columns
    ]
    payload = {
        "label": label,
        "runs": manifest[manifest_cols].sort_values(manifest_cols).to_dict("records") if manifest_cols else [],
        "extra": _cache_jsonable(extra or {}),
    }
    text = json.dumps(payload, sort_keys=True, ensure_ascii=True)
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:16]


def load_or_build_startw_cached_frames(label, manifest, *, extra, builder):
    key = startw_cache_key(manifest, label=label, extra=extra)
    prompt_path = CONTROLLED_STARTW_CACHE_DIR / f"{label}_{key}_prompt.parquet"
    menu_path = CONTROLLED_STARTW_CACHE_DIR / f"{label}_{key}_menu.parquet"
    if CONTROLLED_STARTW_USE_CACHE and not CONTROLLED_STARTW_FORCE_REBUILD_CACHE and prompt_path.exists() and menu_path.exists():
        print(f"Loaded cached {label} frames: {prompt_path.name}, {menu_path.name}")
        return pd.read_parquet(prompt_path), pd.read_parquet(menu_path)
    prompt_df, menu_df = builder()
    prompt_df.to_parquet(prompt_path, index=False)
    menu_df.to_parquet(menu_path, index=False)
    print(f"Wrote cached {label} frames: {prompt_path.name}, {menu_path.name}")
    return prompt_df, menu_df


def load_startw_manifest(
    *,
    data_sources,
    models=CONTROLLED_STARTW_MODELS,
    log_root=LOG_ROOT,
):
    """Discover completed controlled Start(w) target-string runs, filtered before latest-run selection."""
    manifest = discover_eval_runs(log_root=log_root, require_prompt_metrics=True)
    if manifest.empty:
        return manifest

    manifest = _add_target_string_config_columns(manifest)
    manifest = manifest[
        manifest["config"].map(
            lambda c: isinstance(c, dict)
            and c.get("do_decoding") is True
            and c.get("decoding_mapping") == "target_string"
        )
    ].copy()
    manifest = manifest[manifest["config"].map(_matches_controlled_startw_config)].copy()
    manifest = manifest[
        manifest["target_string_scoring_mode"].eq(CONTROLLED_STARTW_SCORING_MODE)
        & manifest["data_source"].isin(list(data_sources))
        & manifest["decoding_lens"].fillna("raw_logitlens").eq("raw_logitlens")
    ].copy()
    if manifest.empty:
        return manifest.reset_index(drop=True)

    manifest["display_model_name"] = manifest["model_name"].map(synthetic_model_display_name)
    manifest["translation_target_lang"] = manifest["data_source"].map(translation_target_from_data_source)
    if models is not None:
        manifest = manifest[manifest["display_model_name"].isin(list(models))].copy()
    if manifest.empty:
        return manifest.reset_index(drop=True)

    manifest = select_latest_unique_runs(
        manifest,
        extra_keys=[
            "data_source",
            "target_string_scoring_mode",
            "synthetic_root",
            "synthetic_langs_key",
            "max_prompts",
            "max_prompts_per_lang",
        ],
    )
    order = {name: idx for idx, name in enumerate(CONTROLLED_STARTW_MODELS)}
    manifest["display_model_sort"] = manifest["display_model_name"].map(order).fillna(len(order)).astype(int)
    return manifest.sort_values(
        ["data_source", "display_model_sort", "display_model_name", "run_mtime_ns", "exp_id"]
    ).reset_index(drop=True)


## Load Start-Token Runs

This notebook only loads runs whose config has `target_string_scoring_mode == "start_tokens_only"`. Adjust the variables in the setup cell to limit models, translation targets, source prompt languages, or run-id prefixes before expanding the large menu artifacts.


In [ ]:
startw_data_sources = [
    *CONTROLLED_STARTW_COPY_CLOZE_DATA_SOURCES,
    *[f"translation_to_{lang}" for lang in CONTROLLED_STARTW_TRANSLATION_TARGET_LANGS],
]

manifest = load_startw_manifest(data_sources=startw_data_sources)

print(f"start-token manifest rows: {len(manifest):,}")
if manifest.empty:
    print("No completed start_tokens_only runs matched the current filters.")
else:
    print(
        manifest[[
            "data_source",
            "display_model_name",
            "exp_id",
            "target_string_scoring_mode",
        ]]
        .sort_values(["data_source", "display_model_name", "exp_id"])
        .to_string(index=False)
    )


In [ ]:
copy_cloze_manifest = manifest[manifest["data_source"].isin(CONTROLLED_STARTW_COPY_CLOZE_DATA_SOURCES)].copy()

def build_copy_cloze_plot_data():
    """Load normalized prompt and menu tables for copy/cloze plots."""
    prompt_df = load_synthetic_target_string_prompt_metrics(copy_cloze_manifest)
    prompt_df = add_normalized_layers(prompt_df)
    menu_df = load_target_string_menu_dataframe(
        copy_cloze_manifest,
        data_sources=CONTROLLED_STARTW_COPY_CLOZE_DATA_SOURCES,
        models=CONTROLLED_STARTW_MODELS,
        prompt_langs=CONTROLLED_STARTW_COPY_CLOZE_LANGS,
        target_langs=CONTROLLED_STARTW_COPY_CLOZE_MENU_LANGS,
        include_values=CONTROLLED_STARTW_START_TOKEN_INCLUDE_VALUES,
    )
    menu_df = add_normalized_layers(menu_df)
    return prompt_df, menu_df


copy_cloze_prompt_df, copy_cloze_menu_df = load_or_build_startw_cached_frames(
    "copy_cloze",
    copy_cloze_manifest,
    extra={
        "data_sources": CONTROLLED_STARTW_COPY_CLOZE_DATA_SOURCES,
        "models": CONTROLLED_STARTW_MODELS,
        "prompt_langs": CONTROLLED_STARTW_COPY_CLOZE_LANGS,
        "target_langs": CONTROLLED_STARTW_COPY_CLOZE_MENU_LANGS,
        "include_values": CONTROLLED_STARTW_START_TOKEN_INCLUDE_VALUES,
        "value_col": CONTROLLED_STARTW_START_TOKEN_VALUE_COL,
    },
    builder=build_copy_cloze_plot_data,
)

print(f"copy/cloze prompt rows: {len(copy_cloze_prompt_df):,}")
print(f"copy/cloze menu rows:   {len(copy_cloze_menu_df):,}")
if not copy_cloze_manifest.empty:
    print("selected copy/cloze runs by model:")
    print(
        copy_cloze_manifest
        .groupby(["data_source", "display_model_name"], dropna=False)
        .size()
        .rename("runs")
        .reset_index()
        .sort_values(["data_source", "display_model_name"])
        .to_string(index=False)
    )
if not copy_cloze_menu_df.empty:
    print("menu language grid coverage:")
    print(
        copy_cloze_menu_df[["data_source", "display_model_name", "prompt_lang", "tgt_lang"]]
        .drop_duplicates()
        .groupby(["data_source", "display_model_name", "prompt_lang"], dropna=False)
        .size()
        .rename("target_langs")
        .reset_index()
        .pivot_table(
            index=["data_source", "display_model_name"],
            columns="prompt_lang",
            values="target_langs",
            fill_value=0,
        )
    )


## Copy

Rows are the main-paper models; columns are the fixed prompt-language order. Curves are raw summed Wendler `Start(w)` probability mass for every menu target language unless `CONTROLLED_STARTW_COPY_CLOZE_MENU_LANGS` is set.


In [ ]:
fig = plot_startw_copy_cloze_grid(
    copy_cloze_menu_df,
    copy_cloze_prompt_df,
    data_source="copy",
    models=CONTROLLED_STARTW_MODELS,
    prompt_langs=CONTROLLED_STARTW_COPY_CLOZE_LANGS,
    target_langs=CONTROLLED_STARTW_COPY_CLOZE_MENU_LANGS,
    layer_col="layer",
    value_col=CONTROLLED_STARTW_START_TOKEN_VALUE_COL,
    yscale="linear",
    ylim=(0.0, 1.0),
    title="[Copy task] Start(w) probability across model layer\nRows = models, Columns = prompt language",
)
save_matplotlib_figure_bundle(fig, PAPER_APPENDIX_FIG_DIR / "01_copy_startw")
plt.show()


## Cloze


In [ ]:
fig = plot_startw_copy_cloze_grid(
    copy_cloze_menu_df,
    copy_cloze_prompt_df,
    data_source="cloze",
    models=CONTROLLED_STARTW_MODELS,
    prompt_langs=CONTROLLED_STARTW_COPY_CLOZE_LANGS,
    target_langs=CONTROLLED_STARTW_COPY_CLOZE_MENU_LANGS,
    layer_col="layer",
    value_col=CONTROLLED_STARTW_START_TOKEN_VALUE_COL,
    yscale="linear",
    ylim=(0.0, 1.0),
    title="[Cloze task] Start(w) probability across model layer\nRows = models, Columns = prompt language",
)
save_matplotlib_figure_bundle(fig, PAPER_APPENDIX_FIG_DIR / "02_cloze_startw")
plt.show()


## Translation by Target Language

Rows are the main-paper models; columns are translation target languages. Each panel averages over source prompt languages after optional `CONTROLLED_STARTW_TRANSLATION_PROMPT_LANGS` filtering, and plots every menu target language unless `CONTROLLED_STARTW_TRANSLATION_MENU_LANGS` is set.


In [ ]:
translation_data_sources = [f"translation_to_{lang}" for lang in CONTROLLED_STARTW_TRANSLATION_TARGET_LANGS]
translation_manifest = manifest[manifest["data_source"].isin(translation_data_sources)].copy()
available_translation_targets = set(translation_manifest["translation_target_lang"].dropna())
translation_target_langs = [
    lang for lang in CONTROLLED_STARTW_TRANSLATION_TARGET_LANGS
    if lang in available_translation_targets
]
translation_data_sources = [f"translation_to_{lang}" for lang in translation_target_langs]

translation_selected_manifest = translation_manifest[translation_manifest["data_source"].isin(translation_data_sources)].copy()


def build_translation_plot_data():
    """Load normalized prompt and menu tables for translation plots."""
    prompt_df = load_synthetic_target_string_prompt_metrics(translation_selected_manifest)
    prompt_df = add_normalized_layers(prompt_df)
    menu_df = load_target_string_menu_dataframe(
        translation_manifest,
        data_sources=translation_data_sources,
        models=CONTROLLED_STARTW_MODELS,
        prompt_langs=CONTROLLED_STARTW_TRANSLATION_PROMPT_LANGS,
        target_langs=CONTROLLED_STARTW_TRANSLATION_MENU_LANGS,
        include_values=CONTROLLED_STARTW_START_TOKEN_INCLUDE_VALUES,
    )
    menu_df = add_normalized_layers(menu_df)
    return prompt_df, menu_df


translation_prompt_df, translation_menu_df = load_or_build_startw_cached_frames(
    "translation",
    translation_selected_manifest,
    extra={
        "data_sources": translation_data_sources,
        "models": CONTROLLED_STARTW_MODELS,
        "prompt_langs": CONTROLLED_STARTW_TRANSLATION_PROMPT_LANGS,
        "target_langs": CONTROLLED_STARTW_TRANSLATION_MENU_LANGS,
        "include_values": CONTROLLED_STARTW_START_TOKEN_INCLUDE_VALUES,
        "value_col": CONTROLLED_STARTW_START_TOKEN_VALUE_COL,
    },
    builder=build_translation_plot_data,
)

translation_prompt_lang_order = list(CONTROLLED_STARTW_TRANSLATION_PROMPT_LANGS or CONTROLLED_STARTW_TRANSLATION_PROMPT_COLUMN_LANGS)
translation_all_target_langs = list(translation_target_langs)
translation_all_data_sources = list(translation_data_sources)
translation_all_prompt_langs = [
    lang for lang in translation_prompt_lang_order
    if translation_menu_df.empty or lang in set(translation_menu_df["prompt_lang"].dropna())
]
translation_all_menu_df = translation_menu_df[
    translation_menu_df["data_source"].isin(translation_all_data_sources)
    & translation_menu_df["prompt_lang"].isin(translation_all_prompt_langs)
].copy()
translation_all_prompt_df = translation_prompt_df[
    translation_prompt_df["data_source"].isin(translation_all_data_sources)
    & translation_prompt_df["prompt_lang"].isin(translation_all_prompt_langs)
].copy()

# Filtered variant: exclude translation cases into English and from English,
# but keep English as a menu target candidate curve.
translation_no_english_target_langs = [lang for lang in translation_target_langs if lang != "en"]
translation_no_english_data_sources = [f"translation_to_{lang}" for lang in translation_no_english_target_langs]
translation_no_english_prompt_langs = [lang for lang in translation_all_prompt_langs if lang != "en"]
translation_no_english_menu_df = translation_menu_df[
    translation_menu_df["data_source"].isin(translation_no_english_data_sources)
    & translation_menu_df["prompt_lang"].isin(translation_no_english_prompt_langs)
].copy()
translation_no_english_prompt_df = translation_prompt_df[
    translation_prompt_df["data_source"].isin(translation_no_english_data_sources)
    & translation_prompt_df["prompt_lang"].isin(translation_no_english_prompt_langs)
].copy()

print(f"translation targets: {translation_target_langs}")
print(f"translation all-case source prompts: {translation_all_prompt_langs}")
print(f"translation no-English-case targets: {translation_no_english_target_langs}")
print(f"translation no-English-case source prompts: {translation_no_english_prompt_langs}")
print(f"translation prompt rows: {len(translation_prompt_df):,}")
print(f"translation menu rows:   {len(translation_menu_df):,}")
print(f"translation all-case prompt rows: {len(translation_all_prompt_df):,}")
print(f"translation all-case menu rows:   {len(translation_all_menu_df):,}")
print(f"translation no-English-case prompt rows: {len(translation_no_english_prompt_df):,}")
print(f"translation no-English-case menu rows:   {len(translation_no_english_menu_df):,}")
if not translation_manifest.empty:
    print("selected all-case translation runs by model/target:")
    print(
        translation_manifest[translation_manifest["data_source"].isin(translation_all_data_sources)]
        .groupby(["display_model_name", "translation_target_lang"], dropna=False)
        .size()
        .rename("runs")
        .reset_index()
        .pivot_table(
            index="display_model_name",
            columns="translation_target_lang",
            values="runs",
            fill_value=0,
        )
    )
if not translation_all_menu_df.empty:
    print("all-case translation menu entries with start-token scores:")
    print(
        summarize_target_string_menu_entries(
            translation_all_menu_df,
            group_cols=("display_model_name", "translation_target_lang", "tgt_lang"),
            value_col=CONTROLLED_STARTW_START_TOKEN_VALUE_COL,
        ).to_string(index=False)
    )


In [ ]:
if CONTROLLED_STARTW_START_TOKEN_VALUE_COL not in translation_all_menu_df.columns:
    print(f"{CONTROLLED_STARTW_START_TOKEN_VALUE_COL} is not available in these translation artifacts yet.")
elif translation_all_menu_df[CONTROLLED_STARTW_START_TOKEN_VALUE_COL].notna().sum() == 0:
    print(f"{CONTROLLED_STARTW_START_TOKEN_VALUE_COL} is present but empty; rerun artifacts have not finished/loaded yet.")
else:
    available_rows = translation_all_menu_df[CONTROLLED_STARTW_START_TOKEN_VALUE_COL].notna().sum()
    print(f"Available all-case translation start-token rows: {available_rows:,}")

fig = plot_startw_translation_target_grid(
    translation_all_menu_df,
    translation_all_prompt_df,
    models=CONTROLLED_STARTW_MODELS,
    target_data_langs=translation_all_target_langs,
    prompt_langs=translation_all_prompt_langs,
    target_langs=CONTROLLED_STARTW_TRANSLATION_MENU_LANGS,
    layer_col="layer",
    value_col=CONTROLLED_STARTW_START_TOKEN_VALUE_COL,
    yscale="linear",
    ylim=(0.0, 1.0),
    title="[Translation task] Start(w) probability across model layer\nRows = models, Columns = target translation language (all cases)",
)
save_matplotlib_figure_bundle(fig, PAPER_APPENDIX_FIG_DIR / "03_translation_startw_by_target_all")
plt.show()

if CONTROLLED_STARTW_START_TOKEN_VALUE_COL not in translation_no_english_menu_df.columns:
    print(f"{CONTROLLED_STARTW_START_TOKEN_VALUE_COL} is not available in these filtered translation artifacts yet.")
elif translation_no_english_menu_df[CONTROLLED_STARTW_START_TOKEN_VALUE_COL].notna().sum() == 0:
    print(f"{CONTROLLED_STARTW_START_TOKEN_VALUE_COL} is present but empty for the no-English-case subset.")
else:
    available_rows = translation_no_english_menu_df[CONTROLLED_STARTW_START_TOKEN_VALUE_COL].notna().sum()
    print(f"Available no-English-case translation start-token rows: {available_rows:,}")

HIDE_TRANSLATION_TARGET_NO_ENGLISH_TITLE = True

fig = plot_startw_translation_target_grid(
    translation_no_english_menu_df,
    translation_no_english_prompt_df,
    models=CONTROLLED_STARTW_MODELS,
    target_data_langs=translation_no_english_target_langs,
    prompt_langs=translation_no_english_prompt_langs,
    target_langs=CONTROLLED_STARTW_TRANSLATION_MENU_LANGS,
    layer_col="layer",
    value_col=CONTROLLED_STARTW_START_TOKEN_VALUE_COL,
    yscale="linear",
    ylim=(0.0, 1.0),
    title=None if HIDE_TRANSLATION_TARGET_NO_ENGLISH_TITLE else "[Translation task] Start(w) probability across model layer\nRows = models, Columns = target translation language (English cases excluded)",
    grid_left=0.125,
    grid_right=0.940,
    grid_wspace=0.025,
    ylabel_x=0.033,
)
save_matplotlib_figure_bundle(fig, PAPER_MAIN_FIG_DIR / "03_translation_startw_by_target_no_en")
plt.show()


### Translation by Source Prompt Language

Same translation runs as above, but columns are source prompt languages. The first plot includes all translation cases; the second excludes `translation_to_en` and English source prompts. English can still appear as a menu target candidate curve in both.


In [ ]:
print(f"all-case translation source prompt columns: {translation_all_prompt_langs}")

fig = plot_startw_translation_source_grid(
    translation_all_menu_df,
    translation_all_prompt_df,
    models=CONTROLLED_STARTW_MODELS,
    prompt_langs=translation_all_prompt_langs,
    target_data_langs=translation_all_target_langs,
    target_langs=CONTROLLED_STARTW_TRANSLATION_MENU_LANGS,
    layer_col="layer",
    value_col=CONTROLLED_STARTW_START_TOKEN_VALUE_COL,
    yscale="linear",
    ylim=(0.0, 1.0),
    title="[Translation task] Start(w) probability across model layer\nRows = models, Columns = source language (all cases)",
)
save_matplotlib_figure_bundle(fig, PAPER_APPENDIX_FIG_DIR / "04_translation_startw_by_source_all")
plt.show()

print(f"no-English-case translation source prompt columns: {translation_no_english_prompt_langs}")

fig = plot_startw_translation_source_grid(
    translation_no_english_menu_df,
    translation_no_english_prompt_df,
    models=CONTROLLED_STARTW_MODELS,
    prompt_langs=translation_no_english_prompt_langs,
    target_data_langs=translation_no_english_target_langs,
    target_langs=CONTROLLED_STARTW_TRANSLATION_MENU_LANGS,
    layer_col="layer",
    value_col=CONTROLLED_STARTW_START_TOKEN_VALUE_COL,
    yscale="linear",
    ylim=(0.0, 1.0),
    title="[Translation task] Start(w) probability across model layer\nRows = models, Columns = source language (English cases excluded)",
)
save_matplotlib_figure_bundle(fig, PAPER_APPENDIX_FIG_DIR / "05_translation_startw_by_source_no_en")
plt.show()
